indcpa.py

In [2]:
import os
import random
from abc import ABC, abstractmethod
from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305


class Cipher_(ABC):

    @abstractmethod
    def keygen(self) -> bytes:
        pass

    @abstractmethod
    def enc(self, key: bytes, text: bytes) -> bytes:
        pass

    @abstractmethod
    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        pass


class INDCPA_Adv(ABC):

    @abstractmethod
    def choose(self, oracle: callable):
        pass

    @abstractmethod
    def guess(self, oracle: callable, ciphertext: bytes):
        pass


def random_bit():
    return random.randint(0, 1)


def IND_CPA(C: Cipher_, A: INDCPA_Adv):

    k = C.keygen()

    enc_oracle = lambda ptxt: C.enc(k, ptxt)

    m0, m1 = A.choose(enc_oracle)

    assert len(m0) == len(m1)

    b = random_bit()

    if b == 0:
        c = C.enc(k, m0)
    else:
        c = C.enc(k, m1)

    b_guess = A.guess(enc_oracle, c)

    return b == b_guess


class IdentityCipher(Cipher_):

    def keygen(self):
        return b''

    def enc(self, key, text):
        return text

    def dec(self, key, ciphertext):
        return ciphertext


class ChaChaCipher(Cipher_):

    def keygen(self):
        return ChaCha20Poly1305.generate_key()

    def enc(self, key, text):

        nonce = os.urandom(12)
        cipher = ChaCha20Poly1305(key)

        return nonce + cipher.encrypt(nonce, text, None)

    def dec(self, key, ciphertext):

        nonce = ciphertext[:12]
        ct = ciphertext[12:]

        cipher = ChaCha20Poly1305(key)

        return cipher.decrypt(nonce, ct, None)


class IdentityAdv(INDCPA_Adv):

    def choose(self, oracle):

        self.m0 = b"A" * 16
        self.m1 = b"B" * 16

        return self.m0, self.m1

    def guess(self, oracle, ciphertext):

        if ciphertext == self.m0:
            return 0
        else:
            return 1


class RandomAdv(INDCPA_Adv):

    def choose(self, oracle):

        m0 = b"A" * 16
        m1 = b"B" * 16

        return m0, m1

    def guess(self, oracle, ciphertext):

        return random_bit()


def run_experiment(cipher, adversary, trials=1000):

    wins = 0

    for _ in range(trials):
        if IND_CPA(cipher, adversary):
            wins += 1

    success_rate = wins / trials

    print("Success rate:", success_rate)


if __name__ == "__main__":

    run_experiment(IdentityCipher(), IdentityAdv())

    run_experiment(ChaChaCipher(), RandomAdv())

ModuleNotFoundError: No module named '_cffi_backend'

thread '<unnamed>' panicked at /usr/share/cargo/registry/pyo3-0.20.2/src/err/mod.rs:788:5:
Python API call failed


PanicException: Python API call failed

indcpa_attck.py

In [ ]:
import os
import random
from abc import ABC, abstractmethod
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

#define modelo de cifra (copiado do guião)
class Cipher_(ABC):

    @abstractmethod 
    def keygen(self) -> bytes:
        pass

    @abstractmethod
    def enc(self, key: bytes, text: bytes) -> bytes:
        pass

    @abstractmethod
    def dec(self, key: bytes, ciphertext: bytes) -> bytes:
        pass

#classe do adversario
class INDCPA_Adv(ABC):

    @abstractmethod # duas fases: escolher mensagem
    def choose(self, oracle: callable):
        pass

    @abstractmethod # e advinhar qual foi a cifra
    def guess(self, oracle: callable, ciphertext: bytes):
        pass


def random_bit(): #bit secreto do jogo que é 0 ou 1
    return random.randint(0, 1)


def IND_CPA(C: Cipher_, A: INDCPA_Adv): #implementação do jogo do guiao

    k = C.keygen() # gera chave secreta

    enc_oracle = lambda ptxt: C.enc(k, ptxt) # função que cifra mensagens com a chave k

    m0, m1 = A.choose(enc_oracle)  # adversario escolhe mensagens

    assert len(m0) == len(m1) # mensagem tem de ter mm tamanho

    b = random_bit() # bit secreto

    if b == 0: #cifrar mensagem desafio
        c = C.enc(k, m0)
    else:
        c = C.enc(k, m1)

    b_guess = A.guess(enc_oracle, c) # player tenta adivinhar

    return b == b_guess # ve se ganhou


# -------------------------
# CIFRA ECB (VULNERÁVEL)
# -------------------------

class ECBCipher(Cipher_): # AES em modo ECB

    def keygen(self):  # gera chave random
        return os.urandom(16)

    def enc(self, key, text): # cifrar

        cipher = Cipher(algorithms.AES(key), modes.ECB())
        encryptor = cipher.encryptor()

        return encryptor.update(text) + encryptor.finalize()

    def dec(self, key, ciphertext): #decifrar

        cipher = Cipher(algorithms.AES(key), modes.ECB())
        decryptor = cipher.decryptor()

        return decryptor.update(ciphertext) + decryptor.finalize()


# ADVERSÁRIO

class ECBAttackAdv(INDCPA_Adv):

    def choose(self, oracle): #escolhe duas mensagens do mesmo tamanho

        # m0 tem dois blocos iguais
        # m1 tem dois blocos diferentes
        self.m0 = b"A" * 16 + b"A" * 16
        self.m1 = b"A" * 16 + b"B" * 16

        return self.m0, self.m1

    def guess(self, oracle, ciphertext): #tenta dar guess

        # divide ciphertext em blocos de 16 bytes
        c1 = ciphertext[:16]
        c2 = ciphertext[16:32]

        # se os blocos forem iguais veio de m0
        if c1 == c2:
            return 0
        else:
            return 1


# teste

def run_experiment(cipher, adversary, trials=1000):

    wins = 0

    for _ in range(trials):

        adv = adversary()   # novo adversário por execução

        if IND_CPA(cipher, adv):
            wins += 1

    success_rate = wins / trials

    print("Ratio de Sucesso:", success_rate)


if __name__ == "__main__":

    run_experiment(ECBCipher(), ECBAttackAdv)

ModuleNotFoundError: No module named '_cffi_backend'

thread '<unnamed>' panicked at /usr/share/cargo/registry/pyo3-0.20.2/src/err/mod.rs:788:5:
Python API call failed
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace


PanicException: Python API call failed